# Game of Thrones Death Prediction — Character Survival Modeling

This notebook trains and evaluates five classifiers on the task of predicting whether a Game of Thrones character is killed during the course of the show.

**Target column:** `is_killed` (1 = character is killed, 0 = character survives)

**Data source:** `reports/characters.parquet` — the cleaned, one-hot encoded characters DataFrame produced by `src.data`.

Models trained:
1. Decision Tree
2. Random Forest
3. Support Vector Classifier (SVC)
4. Naive Bayes
5. Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))
from src.models import split, evaluate

## Load Data

In [ ]:
characters = pd.read_parquet("../reports/characters.parquet")
print(f"Shape: {characters.shape}")
characters.head()

In [ ]:
characters.dtypes

## Baseline Classifier

Before training any models I establish a majority-class baseline. The character dataset is nearly balanced, so the baseline is around 50%.

In [ ]:
class_counts = characters['is_killed'].value_counts()
character_baseline = class_counts.max() / len(characters)

class_counts.plot.bar()
plt.ylabel('Number of Characters')
plt.xlabel('is_killed')
plt.title('Class Balance — is_killed')
plt.tight_layout()
plt.show()

print(f"Majority-class baseline accuracy: {character_baseline:.4f}")

## Prepare Features

Drop the character name column (not a feature), set the target, and split into train / validation / test sets using the `split` helper from `src.models`.

In [ ]:
TARGET = "is_killed"
df = characters.drop(columns=["character_name"], errors="ignore")

X_train, X_val, X_test, y_train, y_val, y_test = split(df, TARGET, random_state=42)

print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

## 1. Decision Tree

I first trained a decision tree. I experimented with several hyperparameter combinations and found that `criterion='entropy'` with `splitter='random'` gave the best validation accuracy of ~0.68. The best model achieved a test accuracy of 0.75.

In [ ]:
dt = DecisionTreeClassifier(criterion='entropy', splitter='random', random_state=42)
dt.fit(X_train, y_train)

val_acc  = metrics.accuracy_score(y_val,  dt.predict(X_val))
test_acc = metrics.accuracy_score(y_test, dt.predict(X_test))

print(f"Decision Tree — val accuracy: {val_acc:.4f}  |  test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test, dt.predict(X_test), zero_division=0))

In [ ]:
# Visualise the tree structure
fig, ax = plt.subplots(figsize=(14, 6))
_ = plot_tree(dt, ax=ax, feature_names=X_train.columns, filled=True, max_depth=3)
plt.title("Decision Tree (max_depth=3 shown)")
plt.tight_layout()
plt.show()

## 2. Random Forest

The random forest outperformed the decision tree. I found that `criterion='entropy'` with `n_estimators=150` achieved the best validation accuracy of ~0.68. The best model achieved a test accuracy of 0.80.

In [ ]:
# Use the same train/val/test split throughout
X_train_rf, X_val_rf, X_test_rf, y_train_rf, y_val_rf, y_test_rf = split(
    df, TARGET, random_state=42
)

forest = RandomForestClassifier(criterion='entropy', n_estimators=150, random_state=42)
forest.fit(X_train_rf, y_train_rf)

val_acc  = metrics.accuracy_score(y_val_rf,  forest.predict(X_val_rf))
test_acc = metrics.accuracy_score(y_test_rf, forest.predict(X_test_rf))

print(f"Random Forest — val accuracy: {val_acc:.4f}  |  test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test_rf, forest.predict(X_test_rf), zero_division=0))

### Feature importances

Looking at the top features selected by the random forest gives useful insight into what drives character survival.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X_train_rf.columns)
top_features = importances.sort_values(ascending=False).head(15)

top_features.plot.barh()
plt.xlabel('Feature Importance')
plt.title('Top 15 Features — Random Forest (Character Survival)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Support Vector Classifier

I looked at the correlation matrix as a sanity check before training an SVC. I didn't find any suspiciously high correlations, which suggests each feature may be providing unique information.

I manually tested several kernel/hyperparameter combinations (grid search was prohibitively slow). The best results came from a polynomial kernel with `degree=3`, `C=10`, `coef0=1`, which achieved a validation accuracy of ~0.67 and a test accuracy of ~0.67.

In [ ]:
# Correlation matrix — eyeball check for highly correlated features
corr = df.drop(columns=[TARGET]).corr()
print("Top correlations (head):")
corr.abs().unstack().sort_values(ascending=False).drop_duplicates().head(20)

In [ ]:
X_train_svc, X_val_svc, X_test_svc, y_train_svc, y_val_svc, y_test_svc = split(
    df, TARGET, random_state=42
)

svc = Pipeline([
    ("scaler",  StandardScaler()),
    ("svc_clf", SVC(kernel="poly", C=10, coef0=1, degree=3, random_state=42)),
])
svc.fit(X_train_svc, y_train_svc)

val_acc  = metrics.accuracy_score(y_val_svc,  svc.predict(X_val_svc))
test_acc = metrics.accuracy_score(y_test_svc, svc.predict(X_test_svc))

print(f"SVC — val accuracy: {val_acc:.4f}  |  test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test_svc, svc.predict(X_test_svc), zero_division=0))

## 4. Naive Bayes

Multinomial Naive Bayes requires non-negative features, which holds for this one-hot encoded DataFrame.

In [ ]:
X_train_nb, X_val_nb, X_test_nb, y_train_nb, y_val_nb, y_test_nb = split(
    df, TARGET, random_state=42
)

nb = make_pipeline(MultinomialNB())
nb.fit(X_train_nb, y_train_nb)

test_acc = metrics.accuracy_score(y_test_nb, nb.predict(X_test_nb))
print(f"Naive Bayes — test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test_nb, nb.predict(X_test_nb), zero_division=0))

## 5. Logistic Regression

Looking at a scatter of the training labels against the index shows how much overlap there is between the two classes — the data is far from linearly separable, which limits how well logistic regression can perform.

In [ ]:
X_train_lg, X_val_lg, X_test_lg, y_train_lg, y_val_lg, y_test_lg = split(
    df, TARGET, random_state=42
)

lgr = LogisticRegression(max_iter=1000, random_state=42)
lgr.fit(X_train_lg, y_train_lg)

test_acc = metrics.accuracy_score(y_test_lg, lgr.predict(X_test_lg))
print(f"Logistic Regression — test accuracy: {test_acc:.4f}")

In [ ]:
print(metrics.classification_report(y_test_lg, lgr.predict(X_test_lg), zero_division=0))

In [ ]:
plt.scatter(X_train_lg.index.values, y_train_lg, alpha=0.5)
plt.xlabel('Sample Index')
plt.ylabel('is_killed')
plt.title('Training Labels — Character Survival\n(overlap shows limited linear separability)')
plt.tight_layout()
plt.show()

## Results Summary

All five models evaluated against the held-out test set, compared to the majority-class baseline.

In [ ]:
results = {
    "Baseline (majority class)": character_baseline,
    "Decision Tree":             metrics.accuracy_score(y_test,     dt.predict(X_test)),
    "Random Forest":             metrics.accuracy_score(y_test_rf,  forest.predict(X_test_rf)),
    "SVC":                       metrics.accuracy_score(y_test_svc, svc.predict(X_test_svc)),
    "Naive Bayes":               metrics.accuracy_score(y_test_nb,  nb.predict(X_test_nb)),
    "Logistic Regression":       metrics.accuracy_score(y_test_lg,  lgr.predict(X_test_lg)),
}

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Test Accuracy'])
results_df = results_df.sort_values('Test Accuracy', ascending=False)

display(results_df)

results_df['Test Accuracy'].plot.barh(color=['grey'] + ['steelblue'] * 5)
plt.xlabel('Accuracy')
plt.title('Character Survival — Model Comparison')
plt.axvline(character_baseline, color='grey', linestyle='--', label='Baseline')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Conclusion

The **Random Forest** achieved the highest test accuracy (~0.80), comfortably beating the majority-class baseline of ~0.50. The Decision Tree came second (~0.75). The SVC and Logistic Regression both scored around 0.67, while Naive Bayes performed the worst on this dataset.

The random forest's top predictive features were screen time and a handful of relational features (whether the character has served others, has been served, etc.), which makes intuitive sense: main characters with more screen time are both more memorable and — as the story demands — more likely to die dramatically.

The scene-level modeling continuation is in `03_scene_modeling.ipynb`.